# Task 5: Circuit generation functions, `bind`, and `num_params`

Walks through every new function added in Task 5. Focus is on physics correctness checks — things unit tests either verify numerically but don't display, or explicitly leave for human review.

In [3]:
import numpy as np
from qaravan.core.circuits import Circuit 
from qaravan.applications.circuit_library import (
    nn_pairs, ghz_circuit,
    rx_layer, ry_layer, rz_layer,
    rxx_layer, ryy_layer, rzz_layer,
)
from qaravan.core.gates import H, CNOT, RX, RY, RZ, RXX, RYY, RZZ

## `nn_pairs` — nearest-neighbor skeleton

In [4]:
print("open chain n=5:", nn_pairs(5))
print("periodic  n=5:", nn_pairs(5, periodic=True))
print("n=1 (empty):",  nn_pairs(1))

open chain n=5: [[0, 1], [1, 2], [2, 3], [3, 4]]
periodic  n=5: [[0, 1], [1, 2], [2, 3], [3, 4], [4, 0]]
n=1 (empty): []


## `ghz_circuit` — structure

In [5]:
circ = ghz_circuit(4)
print(f"num_sites={circ.num_sites}, len={len(circ)}")
for g in circ.gates:
    print(f"  {g.name:6s} {g.indices}")

num_sites=4, len=4
  H      [0]
  CNOT   [0, 1]
  CNOT   [1, 2]
  CNOT   [2, 3]


## `ghz_circuit` — physics check

Apply to $|000\rangle$. Should give $(|000\rangle + |111\rangle)/\sqrt{2}$.
Only indices 0 and 7 should be nonzero at $1/\sqrt{2}$.

In [6]:
n = 3
psi_in = np.zeros(2**n); psi_in[0] = 1.0
psi_out = ghz_circuit(n).to_matrix() @ psi_in

print("Output amplitudes (nonzero only):")
for i, a in enumerate(psi_out):
    if abs(a) > 1e-12:
        bitstring = format(i, f'0{n}b')
        print(f"  |{bitstring}⟩: {a:.6f}")

print(f"\nNorm: {np.linalg.norm(psi_out):.12f}  (should be 1)")
print(f"Equal weights: {np.isclose(abs(psi_out[0]), abs(psi_out[-1]))}  (should be True)")

Output amplitudes (nonzero only):
  |000⟩: 0.707107+0.000000j
  |111⟩: 0.707107+0.000000j

Norm: 1.000000000000  (should be 1)
Equal weights: True  (should be True)


## Rotation layers — structure and params

In [7]:
params = [0.3, 0.7, 1.2, 2.0]
circ = rx_layer(4, params=params)
print("rx_layer with explicit params:")
for g in circ.gates:
    print(f"  {g}")

rx_layer with explicit params:
  RX([0], 0.3)
  RX([1], 0.7)
  RX([2], 1.2)
  RX([3], 2)


In [8]:
# Seed reproducibility
c1 = rz_layer(4, seed=42)
c2 = rz_layer(4, seed=42)
c3 = rz_layer(4, seed=99)
same = all(np.isclose(c1[i].params[0], c2[i].params[0]) for i in range(4))
diff = any(not np.isclose(c1[i].params[0], c3[i].params[0]) for i in range(4))
print(f"Same seed → same params: {same}")
print(f"Diff seed → diff params: {diff}")

Same seed → same params: True
Diff seed → diff params: True


## Rotation layers — physics check

`rx_layer(2, params=[0, 0])` should be the identity on 2 qubits.

In [9]:
U = rx_layer(2, params=[0.0, 0.0]).to_matrix()
print("rx_layer at θ=0 equals I₄:", np.allclose(U, np.eye(4)))

rx_layer at θ=0 equals I₄: True


## Two-site layers — structure

In [10]:
skeleton = nn_pairs(4)
circ = rzz_layer(skeleton, params=[0.1, 0.5, 0.9])
print(f"rzz_layer on nn_pairs(4): {len(circ)} gates")
for g in circ.gates:
    print(f"  {g}")

rzz_layer on nn_pairs(4): 3 gates
  RZZ([0, 1], 0.1)
  RZZ([1, 2], 0.5)
  RZZ([2, 3], 0.9)


In [11]:
circ.num_params

3

## `rzz_layer` — physics check

$\text{RZZ}(\theta) = \exp(-i\theta\, Z\otimes Z)$. The diagonal should be
$\bigl(e^{-i\theta},\, e^{+i\theta},\, e^{+i\theta},\, e^{-i\theta}\bigr)$.
Cross-checks the convention established in Task 4.

In [12]:
theta = 0.7
U = rzz_layer([[0, 1]], params=[theta]).to_matrix()
diag = np.diag(U)
expected = np.array([
    np.exp(-1j * theta),
    np.exp(+1j * theta),
    np.exp(+1j * theta),
    np.exp(-1j * theta),
])
print(f"theta = {theta}")
print(f"diagonal:  {np.round(diag, 4)}")
print(f"expected:  {np.round(expected, 4)}")
print(f"match: {np.allclose(diag, expected)}")

theta = 0.7
diagonal:  [0.7648-0.6442j 0.7648+0.6442j 0.7648+0.6442j 0.7648-0.6442j]
expected:  [0.7648-0.6442j 0.7648+0.6442j 0.7648+0.6442j 0.7648-0.6442j]
match: True


## `num_params` and `bind`

Core use case: build a parametric ansatz, inspect `num_params`, then `bind` a parameter vector.

In [13]:
# Build a simple 1D variational ansatz: RZ layer + RZZ layer + RZ layer
n = 3
skel = nn_pairs(n)

ansatz = rz_layer(n, params=[0.0]*n) + rzz_layer(skel, params=[0.0]*len(skel)) + rz_layer(n, params=[0.0]*n)
print(f"Ansatz: {len(ansatz)} gates, num_params = {ansatz.num_params}")
# Expected: 3 + 2 + 3 = 8 gates, 8 params

# Show zero-param circuit
print("\nGates before bind:")
for g in ansatz.gates:
    print(f"  {g}")

Ansatz: 8 gates, num_params = 8

Gates before bind:
  RZ([0], 0)
  RZ([1], 0)
  RZ([2], 0)
  RZZ([0, 1], 0)
  RZZ([1, 2], 0)
  RZ([0], 0)
  RZ([1], 0)
  RZ([2], 0)


In [14]:
# Bind a random parameter vector
rng = np.random.default_rng(0)
theta = rng.uniform(0, 2*np.pi, ansatz.num_params)

bound = ansatz.bind(theta)
print("Gates after bind:")
for g in bound.gates:
    print(f"  {g}")

print(f"\nOriginal still has zero params: {ansatz[0].params[0] == 0.0}")

Gates after bind:
  RZ([0], 4.002)
  RZ([1], 1.695)
  RZ([2], 0.2574)
  RZZ([0, 1], 0.1038)
  RZZ([1, 2], 5.11)
  RZ([0], 5.735)
  RZ([1], 3.812)
  RZ([2], 4.584)

Original still has zero params: True


## `bind` — parameter assignment order

Parameters are consumed **left to right in gate order**, not by site index.
Mixed circuit: `RX(0)`, `H(1)`, `RZZ([1,2])` — `num_params = 2`, `H` is skipped.

In [15]:
mixed = Circuit([RX(0, 0.0), H(1), RZZ([1, 2], 0.0)])
print(f"num_params = {mixed.num_params}  (H does not count)")

bound = mixed.bind([0.5, 1.2])
print(f"RX param after bind: {bound[0].params[0]:.4f}  (expected 0.5)")
print(f"H unchanged: {bound[1].name}")
print(f"RZZ param after bind: {bound[2].params[0]:.4f}  (expected 1.2)")

num_params = 2  (H does not count)
RX param after bind: 0.5000  (expected 0.5)
H unchanged: H
RZZ param after bind: 1.2000  (expected 1.2)


## `bind` — physics check

Start from a zero-angle RX circuit, bind to $\pi/2$, and verify the matrix matches `RX(0, π/2)` directly.

In [16]:
circ = rx_layer(1, params=[0.0])
bound = circ.bind([np.pi / 2])

expected = RX(0, np.pi / 2).matrix
print("bound matrix:")
print(np.round(bound.to_matrix(), 4))
print("\nexpected (RX at π/2 = -iX):")
print(np.round(expected, 4))
print(f"\nmatch: {np.allclose(bound.to_matrix(), expected)}")

bound matrix:
[[0.+0.j 0.-1.j]
 [0.-1.j 0.+0.j]]

expected (RX at π/2 = -iX):
[[0.+0.j 0.-1.j]
 [0.-1.j 0.+0.j]]

match: True


## `bind` — wrong param count

Should raise `AssertionError` immediately.

In [17]:
try:
    rx_layer(3, params=[0.0]*3).bind([1.0, 2.0])  # only 2, needs 3
    print("ERROR: should have raised")
except AssertionError as e:
    print(f"AssertionError raised as expected")

AssertionError raised as expected


## Trotter-step building block

`rzz_layer` on `nn_pairs` is the natural building block for a 1D ZZ Trotter step.
Below: one step of $\exp(-i\delta t\, \sum_i Z_i Z_{i+1})$ on 4 sites, then verify unitarity.

In [18]:
dt = 0.1
n = 4
trotter_step = rzz_layer(nn_pairs(n), params=[dt] * (n - 1))
U = trotter_step.to_matrix()
print(f"Trotter step unitary: {np.allclose(U @ U.conj().T, np.eye(2**n))}")

Trotter step unitary: True
